# Familiarity vs. Answerability: Colab Execution

This notebook orchestrates the preregistered pipeline. It contains no estimators, scientific scoring logic, or claim decisions. Protected endpoints remain closed until their dedicated CLI transactions are available.


In [ ]:
from __future__ import annotations

import hashlib
import json
import os
import shutil
import subprocess
from pathlib import Path

import torch
from google.colab import drive

assert os.environ.get("HF_TOKEN"), "Set HF_TOKEN before model access."
assert torch.cuda.is_available(), "A Colab GPU runtime is required."
gpu = torch.cuda.get_device_properties(0)
disk = shutil.disk_usage("/content")
preflight = {
    "gpu": gpu.name,
    "gpu_gib": round(gpu.total_memory / 2**30, 2),
    "ram_gib": round(os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30, 2),
    "disk_free_gib": round(disk.free / 2**30, 2),
}
assert preflight["gpu_gib"] >= 14 and preflight["disk_free_gib"] >= 40, preflight
preflight


In [ ]:
drive.mount("/content/drive")
REPO = Path("/content/mechanistic-interpretability")
DRIVE_CHECKPOINT_ROOT = Path("/content/drive/MyDrive/fa-study-checkpoints")
DRIVE_CHECKPOINT_ROOT.mkdir(parents=True, exist_ok=True)
LOCAL_ROOT = Path("/content/fa-study-work")
LOCAL_ROOT.mkdir(parents=True, exist_ok=True)
assert REPO.is_dir(), "Clone the pinned repository checkout into /content first."
os.chdir(REPO)
expected_commit = os.environ.get("FA_GIT_COMMIT")
assert expected_commit, "Set FA_GIT_COMMIT to the frozen 40-character commit."
actual_commit = subprocess.check_output(["git", "rev-parse", "HEAD"], text=True).strip()
assert actual_commit == expected_commit, (actual_commit, expected_commit)


## Pinned environment

Install only the core lock. The optional circuit profile is not part of the confirmatory run.


In [ ]:
subprocess.run(["python", "-m", "pip", "install", "-r", "requirements/fa-core.lock"], check=True)
subprocess.run(["python", "-m", "pip", "install", "--no-deps", "-e", "."], check=True)
lock_bytes = Path("requirements/fa-core.lock").read_bytes()
print("fa-core.lock sha256:", hashlib.sha256(lock_bytes).hexdigest())


In [ ]:
CONFIG = "configs/familiarity_answerability_gemma2_2b.json"
ROOT = str(LOCAL_ROOT)

def run_cli(*arguments: str) -> None:
    command = ["python", "-m", "trajectory_extractor.cli", *arguments]
    print(" ".join(command))
    subprocess.run(command, check=True)


## Resumable core sequence

Set the paths below only to verified artifacts from the same run. `fa-materialize-probe-rows` creates compact, provenance-bound evidence from generation, registered activations, exact teacher-forced scores, metadata, and outcomes. Protected rows remain unreadable to selection code.


In [ ]:
run_cli("fa-audit-manifest", "--config", CONFIG, "--root", ROOT, "--manifest", "<verified-manifest>")
run_cli("fa-materialize-probe-rows", "--config", CONFIG, "--root", ROOT, "--namespace", "mechanism_train", "--manifest", "<mechanism-prompt-manifest>", "--metadata-manifest", "<probe-metadata-manifest>", "--shard-id", "mechanism-0000", "--resume")
run_cli("fa-materialize-probe-rows", "--config", CONFIG, "--root", ROOT, "--namespace", "locked_validation", "--manifest", "<validation-prompt-manifest>", "--metadata-manifest", "<probe-metadata-manifest>", "--shard-id", "validation-0000", "--resume")
run_cli("fa-fit-probes", "--config", CONFIG, "--root", ROOT, "--train-rows-manifest", "<mechanism-probe-rows>", "--validation-rows-manifest", "<validation-probe-rows>", "--probe-test-manifest", "<probe-test-prompt-manifest>", "--shard-id", "selection-0000")
run_cli("fa-seal-behavior-test", "--config", CONFIG, "--root", ROOT, "--behavior-test-manifest", "<behavior-test-prompt-manifest>")
run_cli("fa-seal-selection", "--config", CONFIG, "--root", ROOT, "--selection-manifest", "<f2a-selection-manifest>", "--probe-test-manifest", "<probe-test-prompt-manifest>")
run_cli("fa-evaluate-behavior-test", "--config", CONFIG, "--root", ROOT, "--manifest", "<behavior-test-prompt-manifest>", "--shard-id", "behavior-0000")
run_cli("fa-evaluate-probe-test", "--config", CONFIG, "--root", ROOT, "--selection-manifest", "<f2a-selection-manifest>", "--probe-test-manifest", "<probe-test-prompt-manifest>", "--metadata-manifest", "<probe-metadata-manifest>", "--shard-id", "probe-test-0000")
run_cli("fa-build-report", "--config", CONFIG, "--root", ROOT, "--behavior-test-manifest", "<behavior-test-prompt-manifest>", "--probe-test-manifest", "<probe-test-prompt-manifest>", "--selection-manifest", "<f2a-selection-manifest>", "--output", "reports/familiarity_answerability.md")


## Stop conditions

Stop on any pin, hash, audit, OOM, completion, or endpoint-state failure. Run transactions on local Colab storage. After each successful transaction, checkpoint only completed checksum-verified shards to `DRIVE_CHECKPOINT_ROOT`; restore only artifacts that pass their sidecar verification. A runtime interruption resumes from verified shard manifests rather than filenames. The notebook does not treat Google Drive writes as atomic.
